# 20 Actionable Insights from Aadhaar Data

This notebook generates 20 distinct, actionable insights from the cleaned UIDAI datasets. Each insight is designed to uncover specific operational patterns, risks, and opportunities in the Aadhaar ecosystem.

**Methodology:**
1.  **Load Cleaned Data**: Ingest the final, validated datasets from the `final_pipeline_output` directory.
2.  **Pre-process & Aggregate**: Standardize data types, parse dates, and create aggregated DataFrames for daily, monthly, and district-level analysis.
3.  **Generate Insights**: For each of the 20 points, calculate the required metrics and identify the districts or states that match the criteria.
4.  **Visualize & Report**: Display the findings in clear, interpretable tables with explanations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
from scipy.stats import linregress

# --- Configuration ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
ROOT = Path(os.getcwd()).resolve()
DATA_DIR = ROOT / "final_pipeline_output"
LOGS_DIR = DATA_DIR / "logs"
RAW_DATA_DIR = ROOT / "data" # For meta-insights

print(f"Project Root: {ROOT}")
print(f"Data Directory: {DATA_DIR}")

Project Root: D:\exp\uidai-datathon-2026-participation
Data Directory: D:\exp\uidai-datathon-2026-participation\final_pipeline_output


## 1. Load and Pre-process Data

This section loads the cleaned biometric, demographic, and enrolment CSVs, concatenates them, and creates a master analysis table. It also prepares several aggregated views (e.g., daily totals, district totals) for efficient analysis.

In [2]:
def load_data_from_category(category_path: Path) -> pd.DataFrame:
    """Loads and concatenates all CSV files from a given category directory."""
    if not category_path.exists():
        print(f"[WARN] Directory not found: {category_path}")
        return pd.DataFrame()
    
    csv_files = list(category_path.glob("*.csv"))
    if not csv_files:
        print(f"[WARN] No CSV files found in {category_path}")
        return pd.DataFrame()
        
    dfs = [pd.read_csv(file, dtype=str) for file in csv_files]
    return pd.concat(dfs, ignore_index=True)

# Load the data
print("Loading datasets...")
enrolment_df = load_data_from_category(DATA_DIR / "enrolment")
biometric_df = load_data_from_category(DATA_DIR / "biometric")
demographic_df = load_data_from_category(DATA_DIR / "demographic")

# --- Pre-processing ---
print("Pre-processing data...")
dataframes = {'enrolment': enrolment_df, 'biometric': biometric_df, 'demographic': demographic_df}

for name, df in dataframes.items():
    if df.empty: continue
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    for col in df.columns:
        if 'age' in col or 'bio' in col or 'demo' in col:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Create total columns
if not enrolment_df.empty:
    enrolment_df['total_enrolment'] = enrolment_df[['age_0_5', 'age_5_17', 'age_18_greater']].sum(axis=1)
if not biometric_df.empty:
    biometric_df['total_biometric_updates'] = biometric_df[['bio_age_5_17', 'bio_age_17_']].sum(axis=1)
if not demographic_df.empty:
    demographic_df['total_demographic_updates'] = demographic_df[['demo_age_5_17', 'demo_age_17_']].sum(axis=1)

# --- Create Aggregated DataFrames ---
group_cols = ['date', 'state', 'district']
enrol_agg = enrolment_df.groupby(group_cols)['total_enrolment'].sum().reset_index() if not enrolment_df.empty else pd.DataFrame(columns=group_cols)
bio_agg = biometric_df.groupby(group_cols)['total_biometric_updates'].sum().reset_index() if not biometric_df.empty else pd.DataFrame(columns=group_cols)
demo_agg = demographic_df.groupby(group_cols)['total_demographic_updates'].sum().reset_index() if not demographic_df.empty else pd.DataFrame(columns=group_cols)

# Master daily DataFrame
master_df = pd.merge(enrol_agg, bio_agg, on=group_cols, how='outer')
master_df = pd.merge(master_df, demo_agg, on=group_cols, how='outer').fillna(0)
master_df['total_transactions'] = master_df.get('total_enrolment', 0) + master_df.get('total_biometric_updates', 0) + master_df.get('total_demographic_updates', 0)
master_df['total_updates'] = master_df.get('total_biometric_updates', 0) + master_df.get('total_demographic_updates', 0)
master_df = master_df.dropna(subset=['date', 'state', 'district'])

# District-level totals for the entire period
district_totals = master_df.groupby(['state', 'district']).agg(
    total_enrolments=('total_enrolment', 'sum'),
    total_updates=('total_updates', 'sum'),
    total_transactions=('total_transactions', 'sum')
).reset_index()

print("Master and aggregated DataFrames created successfully.")
display(master_df.head())
display(district_totals.head())

Loading datasets...
Pre-processing data...
Master and aggregated DataFrames created successfully.


,date,state,district,total_enrolment,total_biometric_updates,total_demographic_updates,total_transactions,total_updates
0,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,0.0,817.0,422.0,1239.0,1239.0
1,2025-01-03,Andhra Pradesh,Alluri Sitharama Raju,0.0,1993.0,4955.0,6948.0,6948.0
2,2025-01-03,Andhra Pradesh,Anakapalli,0.0,452.0,3164.0,3616.0,3616.0
3,2025-01-03,Andhra Pradesh,Ananthapuramu,0.0,16150.0,10438.0,26588.0,26588.0
4,2025-01-03,Andhra Pradesh,Annamayya,0.0,505.0,3798.0,4303.0,4303.0


,state,district,total_enrolments,total_updates,total_transactions
0,Andaman And Nicobar Islands,Nicobars,1.0,4.0,5.0
1,Andaman And Nicobar Islands,North And Middle Andaman,49.0,6574.0,6623.0
2,Andhra Pradesh,Alluri Sitharama Raju,358.0,22848.0,23206.0
3,Andhra Pradesh,Anakapalli,143.0,13487.0,13630.0
4,Andhra Pradesh,Ananthapuramu,874.0,135008.0,135882.0


### 1️⃣ Aadhaar Service Penetration Density
**What**: How intensively Aadhaar services are used per district.

**Why**: Identifies under-served districts where usage is low relative to state and national norms, suggesting a potential infrastructure or access gap.

In [3]:
print("--- Insight 1: Aadhaar Service Penetration Density ---")
# Calculate state and national medians
state_median_transactions = district_totals.groupby('state')['total_transactions'].transform('median')
national_median_transactions = district_totals['total_transactions'].median()

penetration_density = district_totals.copy()
penetration_density['state_median'] = state_median_transactions
penetration_density['national_median'] = national_median_transactions

# Identify under-served districts (e.g., below 50% of both medians)
under_served_districts = penetration_density[
    (penetration_density['total_transactions'] < 0.5 * penetration_density['state_median']) &
    (penetration_density['total_transactions'] < 0.5 * penetration_density['national_median'])
]

print(f"National Median Transactions per District: {national_median_transactions:,.0f}")
print(f"Found {len(under_served_districts)} potentially under-served districts.")
display(under_served_districts.sort_values('total_transactions').head(10))

--- Insight 1: Aadhaar Service Penetration Density ---
National Median Transactions per District: 88,751
Found 116 potentially under-served districts.


,state,district,total_enrolments,total_updates,total_transactions,state_median,national_median
0,Andaman And Nicobar Islands,Nicobars,1.0,4.0,5.0,3314.0,88751.0
33,Arunachal Pradesh,Leparada,1.0,27.0,28.0,2212.0,88751.0
659,Uttar Pradesh,Mahrajganj,1.0,29.0,30.0,169743.0,88751.0
417,Mizoram,Khawzawl,9.0,25.0,34.0,12283.0,88751.0
528,Sikkim,Mangan,1.0,35.0,36.0,107.0,88751.0
416,Mizoram,Hnahthial,0.0,39.0,39.0,12283.0,88751.0
397,Manipur,Pherzawl,1.0,53.0,54.0,25099.0,88751.0
213,Himachal Pradesh,Lahaul And Spiti,0.0,58.0,58.0,31820.0,88751.0
520,Rajasthan,Phalodi,0.0,71.0,71.0,128967.0,88751.0
40,Arunachal Pradesh,Pakke Kessang,0.0,87.0,87.0,2212.0,88751.0


### 2️⃣ Enrolment–Update Imbalance Score
**What**: Whether a district is stuck in enrolment-heavy or update-heavy mode.

**Why**: Shows the maturity of the Aadhaar lifecycle in a district. Extreme ratios can indicate a misallocation of resources (e.g., too many enrolment centers where update kiosks are needed).

In [4]:
print("\\n--- Insight 2: Enrolment–Update Imbalance Score ---")
imbalance_df = district_totals.copy()
imbalance_df['imbalance_score'] = imbalance_df['total_enrolments'] / (imbalance_df['total_updates'] + 1)

# Identify extremes (e.g., top and bottom 10%)
enrolment_heavy = imbalance_df[imbalance_df['imbalance_score'] > imbalance_df['imbalance_score'].quantile(0.90)]
update_heavy = imbalance_df[imbalance_df['imbalance_score'] < imbalance_df['imbalance_score'].quantile(0.10)]

print(f"Found {len(enrolment_heavy)} enrolment-heavy districts (top 10%).")
display(enrolment_heavy.sort_values('imbalance_score', ascending=False).head(5))

print(f"\\nFound {len(update_heavy)} update-heavy districts (bottom 10%).")
display(update_heavy.sort_values('imbalance_score').head(5))

\n--- Insight 2: Enrolment–Update Imbalance Score ---
Found 72 enrolment-heavy districts (top 10%).


,state,district,total_enrolments,total_updates,total_transactions,imbalance_score
264,Karnataka,Bengaluru Urban,22590.0,0.0,22590.0,22590.0
108,Bihar,Purbi Champaran,14725.0,0.0,14725.0,14725.0
305,Madhya Pradesh,Ashoknagar,3011.0,0.0,3011.0,3011.0
191,Haryana,Gurugram,2636.0,0.0,2636.0,2636.0
199,Haryana,Nuh,1506.0,0.0,1506.0,1506.0


\nFound 72 update-heavy districts (bottom 10%).


,state,district,total_enrolments,total_updates,total_transactions,imbalance_score
416,Mizoram,Hnahthial,0.0,39.0,39.0,0.0
213,Himachal Pradesh,Lahaul And Spiti,0.0,58.0,58.0,0.0
522,Rajasthan,Salumbar,0.0,92.0,92.0,0.0
520,Rajasthan,Phalodi,0.0,71.0,71.0,0.0
40,Arunachal Pradesh,Pakke Kessang,0.0,87.0,87.0,0.0


### 3️⃣ Dormant District Detection
**What**: Districts with long periods of inactivity.

**Why**: Signals potentially broken, closed, or absent enrolment/update centers.

In [5]:
print("\\n--- Insight 3: Dormant District Detection ---")
total_days_in_dataset = (master_df['date'].max() - master_df['date'].min()).days

active_days_df = master_df.groupby(['state', 'district'])['date'].nunique().reset_index(name='active_days')
active_days_df['activity_ratio'] = active_days_df['active_days'] / total_days_in_dataset

# Flag districts with very low activity (e.g., active less than 10% of the time)
dormant_districts = active_days_df[active_days_df['activity_ratio'] < 0.1]

print(f"Total days in dataset range: {total_days_in_dataset}")
print(f"Found {len(dormant_districts)} dormant districts (active < 10% of the time).")
display(dormant_districts.sort_values('active_days').head(10))

\n--- Insight 3: Dormant District Detection ---
Total days in dataset range: 343
Found 33 dormant districts (active < 10% of the time).


,state,district,active_days,activity_ratio
461,Odisha,Nabarangpur,1,0.002915
199,Haryana,Nuh,1,0.002915
234,Jammu And Kashmir,Shopian,1,0.002915
242,Jharkhand,East Singhbum,1,0.002915
77,Assam,Sivasagar,2,0.005831
305,Madhya Pradesh,Ashoknagar,2,0.005831
108,Bihar,Purbi Champaran,3,0.008746
191,Haryana,Gurugram,3,0.008746
264,Karnataka,Bengaluru Urban,3,0.008746
0,Andaman And Nicobar Islands,Nicobars,5,0.014577


### 4️⃣ Daily Throughput Volatility
**What**: How unstable or inconsistent daily processing volume is within a district.

**Why**: High volatility can be an indicator of operational inefficiency, such as staffing shortages, intermittent infrastructure problems, or unpredictable public demand.

In [6]:
print("\\n--- Insight 4: Daily Throughput Volatility ---")
daily_volatility = master_df.groupby(['state', 'district'])['total_transactions'].agg(['mean', 'std']).reset_index()
daily_volatility['coeff_of_variation'] = daily_volatility['std'] / (daily_volatility['mean'] + 1)

# Identify highly volatile districts (e.g., top 10%)
volatility_threshold = daily_volatility['coeff_of_variation'].quantile(0.90)
highly_volatile_districts = daily_volatility[daily_volatility['coeff_of_variation'] > volatility_threshold]

print(f"Found {len(highly_volatile_districts)} districts with high throughput volatility (top 10%).")
display(highly_volatile_districts.sort_values('coeff_of_variation', ascending=False).head(10))

\n--- Insight 4: Daily Throughput Volatility ---
Found 72 districts with high throughput volatility (top 10%).


,state,district,mean,std,coeff_of_variation
106,Bihar,Pashchim Champaran,433.000000,1835.438640,4.229121
423,Mizoram,Serchhip,230.800000,880.928362,3.800381
182,Gujarat,Surendranagar,56.500000,215.079188,3.740508
421,Mizoram,Mamit,274.048780,967.060881,3.515961
414,Mizoram,Aizawl,1021.000000,3479.286450,3.404390
420,Mizoram,Lunglei,513.225000,1715.933124,3.336931
415,Mizoram,Champhai,518.500000,1693.243996,3.259372
132,Chhattisgarh,Kondagaon,2227.317073,7089.631593,3.181608
625,Uttar Pradesh,Bhadohi,343.975610,1085.615666,3.146935
418,Mizoram,Kolasib,333.250000,1045.363579,3.127490


### 5️⃣ Peak Load Stress Index
**What**: Measures how extreme peak transaction days are compared to normal days.

**Why**: A high index shows an inability to handle temporary surges in demand, leading to long queues, system crashes, and poor public experience.

In [7]:
print("\\n--- Insight 5: Peak Load Stress Index ---")
peak_load_df = master_df.groupby(['state', 'district'])['total_transactions'].agg(['max', 'median']).reset_index()
peak_load_df['peak_stress_index'] = peak_load_df['max'] / (peak_load_df['median'] + 1)

# Identify districts with high stress index (e.g., > 5x the median)
high_stress_districts = peak_load_df[peak_load_df['peak_stress_index'] > 5]

print(f"Found {len(high_stress_districts)} districts with a high peak load stress index (>5x median).")
display(high_stress_districts.sort_values('peak_stress_index', ascending=False).head(10))

\n--- Insight 5: Peak Load Stress Index ---
Found 688 districts with a high peak load stress index (>5x median).


,state,district,max,median,peak_stress_index
262,Karnataka,Bengaluru Rural,1654.0,2.0,551.333333
106,Bihar,Pashchim Champaran,9642.0,19.0,482.100000
421,Mizoram,Mamit,5548.0,23.0,231.166667
423,Mizoram,Serchhip,5487.0,26.5,199.527273
415,Mizoram,Champhai,9812.0,62.0,155.746032
420,Mizoram,Lunglei,10252.0,67.5,149.664234
418,Mizoram,Kolasib,5739.0,45.5,123.419355
414,Mizoram,Aizawl,21803.0,189.0,114.752632
120,Chandigarh,Chandigarh,44127.0,530.0,83.101695
134,Chhattisgarh,Mahasamund,89053.0,1075.0,82.763011


### 6️⃣ Chronic Under-Utilization Zones
**What**: Districts where transaction volumes are consistently low.

**Why**: Highlights potentially wasted infrastructure or centers that are poorly located or inaccessible.

In [8]:
print("\\n--- Insight 6: Chronic Under-Utilization Zones ---")
# Calculate state 25th percentile for daily transactions
state_25th_percentile = master_df.groupby('state')['total_transactions'].transform('quantile', 0.25)
daily_with_percentile = master_df.copy()
daily_with_percentile['state_25th_percentile'] = state_25th_percentile

# Find districts where most of their active days are below the state's 25th percentile
underutilized_check = daily_with_percentile[daily_with_percentile['total_transactions'] < daily_with_percentile['state_25th_percentile']]
days_below_percentile = underutilized_check.groupby(['state', 'district']).size().reset_index(name='days_below_25th')
total_active_days = master_df.groupby(['state', 'district']).size().reset_index(name='total_days')

merged_utilization = pd.merge(days_below_percentile, total_active_days, on=['state', 'district'])
merged_utilization['underutilization_ratio'] = merged_utilization['days_below_25th'] / merged_utilization['total_days']

# Flag if > 80% of active days are under-utilized
chronic_underutilization = merged_utilization[merged_utilization['underutilization_ratio'] > 0.8]

print(f"Found {len(chronic_underutilization)} chronically under-utilized districts.")
display(chronic_underutilization.sort_values('underutilization_ratio', ascending=False).head(10))

\n--- Insight 6: Chronic Under-Utilization Zones ---
Found 78 chronically under-utilized districts.


,state,district,days_below_25th,total_days,underutilization_ratio
0,Andaman And Nicobar Islands,Nicobars,5,5,1.0
464,Rajasthan,Balotra,12,12,1.0
246,Karnataka,Chamarajanagar,41,41,1.0
314,Madhya Pradesh,Pandhurna,36,36,1.0
334,Maharashtra,Ahilyanagar,8,8,1.0
342,Maharashtra,Dharashiv,41,41,1.0
345,Maharashtra,Gondia,35,35,1.0
373,Manipur,Pherzawl,18,18,1.0
390,Mizoram,Hnahthial,17,17,1.0
391,Mizoram,Khawzawl,15,15,1.0


### 7️⃣ District Fragmentation Signal
**What**: The same canonical district appearing with multiple different spellings or names in the raw data.

**Why**: This is a direct measure of administrative data chaos. High fragmentation can corrupt analytics and signals governance risks.

In [9]:
print("\\n--- Insight 7: District Fragmentation Signal ---")
# This insight requires the raw data to find aliases
raw_enrolment = load_data_from_category(RAW_DATA_DIR / "enrolment")
raw_biometric = load_data_from_category(RAW_DATA_DIR / "biometric")
raw_demographic = load_data_from_category(RAW_DATA_DIR / "demographic")
raw_df = pd.concat([raw_enrolment, raw_biometric, raw_demographic], ignore_index=True)

# We need the mapping from the cleaning pipeline (insights.ipynb) or re-run a simplified version
# For now, we'll just count unique raw district names per *cleaned* district name
if not raw_df.empty:
    # Get cleaned district names from our master table
    cleaned_districts = master_df[['state', 'district']].drop_duplicates()
    
    # This is a simplified proxy. A full implementation would map raw->clean.
    # Here we count unique raw district names within a state.
    fragmentation = raw_df.groupby('state')['district'].nunique().reset_index(name='raw_district_alias_count')
    
    print("Note: This is a proxy. A full implementation needs a raw-to-clean mapping.")
    print("High alias count within a state suggests fragmentation.")
    display(fragmentation.sort_values('raw_district_alias_count', ascending=False).head(10))
else:
    print("Raw data not found. Skipping fragmentation analysis.")

\n--- Insight 7: District Fragmentation Signal ---
Note: This is a proxy. A full implementation needs a raw-to-clean mapping.
High alias count within a state suggests fragmentation.


,state,raw_district_alias_count
54,Uttar Pradesh,96
61,West Bengal,66
32,Madhya Pradesh,61
27,Karnataka,56
33,Maharashtra,54
40,Odisha,49
7,Bihar,48
3,Andhra Pradesh,47
49,Tamil Nadu,46
47,Rajasthan,46


### 8️⃣ State Dependency Risk
**What**: States that rely on a very small number of districts for the majority of their Aadhaar transactions.

**Why**: Shows a fragile or overly centralized service distribution. An outage in one of these key districts could cripple the state's Aadhaar services.

In [10]:
print("\\n--- Insight 8: State Dependency Risk ---")
state_totals = district_totals.groupby('state')['total_transactions'].sum().reset_index(name='state_total')
district_dependency = pd.merge(district_totals, state_totals, on='state')
district_dependency['pct_of_state_total'] = 100 * district_dependency['total_transactions'] / district_dependency['state_total']

# Get top 3 districts per state
top_3_districts = district_dependency.sort_values('pct_of_state_total', ascending=False).groupby('state').head(3)
state_dependency_risk = top_3_districts.groupby('state')['pct_of_state_total'].sum().reset_index(name='pct_from_top_3')

# Flag states where top 3 districts account for > 50% of volume
high_risk_states = state_dependency_risk[state_dependency_risk['pct_from_top_3'] > 50]

print(f"Found {len(high_risk_states)} states with high dependency on a few districts.")
display(high_risk_states.sort_values('pct_from_top_3', ascending=False))

\n--- Insight 8: State Dependency Risk ---
Found 14 states with high dependency on a few districts.


,state,pct_from_top_3
0,Andaman And Nicobar Islands,100.000000
5,Chandigarh,100.000000
8,Goa,100.000000
16,Ladakh,100.000000
24,Puducherry,100.000000
27,Sikkim,100.000000
30,The Dadra And Nagar Haveli And Daman And Diu,100.000000
7,Delhi,82.402296
31,Tripura,66.539672
21,Mizoram,62.369641


### 9️⃣ Sudden Volume Shock Detection
**What**: Abrupt surges or drops in daily transaction volume.

**Why**: Can indicate external events like policy changes, local disasters, or technical outages.

In [11]:
print("\\n--- Insight 9: Sudden Volume Shock Detection ---")
# Calculate 7-day rolling mean for each district
master_df['rolling_mean_7d'] = master_df.groupby(['state', 'district'])['total_transactions'].transform(lambda x: x.rolling(7, min_periods=3).mean())
master_df['volume_shock'] = master_df['total_transactions'] - master_df['rolling_mean_7d']
master_df['shock_factor'] = master_df['volume_shock'] / (master_df['rolling_mean_7d'] + 1)

# Identify significant shocks (e.g., > 3x the rolling mean)
significant_shocks = master_df[master_df['shock_factor'].abs() > 3]

print(f"Found {len(significant_shocks)} instances of sudden volume shocks.")
display(significant_shocks.sort_values('shock_factor', ascending=False).head(10))

\n--- Insight 9: Sudden Volume Shock Detection ---
Found 3 instances of sudden volume shocks.


,date,state,district,total_enrolment,total_biometric_updates,total_demographic_updates,total_transactions,total_updates,rolling_mean_7d,volume_shock,shock_factor
18481,2025-08-09,Meghalaya,East Garo Hills,45.0,25.0,106.0,176.0,131.0,35.714286,140.285714,3.821012
18488,2025-08-09,Meghalaya,South West Garo Hills,51.0,20.0,115.0,186.0,135.0,41.142857,144.857143,3.437288
22709,2025-10-09,Meghalaya,Eastern West Khasi Hills,5.0,0.0,72.0,77.0,72.0,17.285714,59.714286,3.265625


### 1️⃣0️⃣ Inter-District Service Substitution
**What**: When a drop in one district's activity coincides with a spike in a neighboring district.

**Why**: Suggests that citizens are traveling across district lines to access services, possibly due to better service quality or lack of local options.

In [13]:
print("\\n--- Insight 10: Inter-District Service Substitution ---")
print("Note: This is a complex analysis requiring geospatial data or a predefined neighbor list.")

# Simplified approach: Correlate all district time series within the same state
# A full implementation would use a neighbor graph.
state_correlations = {}
for state in master_df['state'].unique():
    state_df = master_df[master_df['state'] == state]
    pivot = state_df.pivot_table(index='date', columns='district', values='total_transactions').fillna(0)
    
    if pivot.shape[1] < 2: continue # Need at least 2 districts to correlate
    
    corr_matrix = pivot.corr()
    # Find pairs with strong negative correlation (e.g., < -0.5)
    neg_corr = corr_matrix[corr_matrix < -0.5]
    if not neg_corr.empty:
        state_correlations[state] = neg_corr.unstack().dropna().reset_index()
        state_correlations[state] = state_correlations[state][state_correlations[state]['level_0'] != state_correlations[state]['level_1']]

if state_correlations:
    print("Found potential substitution patterns in the following states (showing one example):")
    example_state = next(iter(state_correlations))
    display(state_correlations[example_state].head())
else:
    print("No strong negative correlations found with this simplified method.")

\n--- Insight 10: Inter-District Service Substitution ---
Note: This is a complex analysis requiring geospatial data or a predefined neighbor list.


ValueError: cannot insert district, already exists

### 1️⃣1️⃣ Update Latency Proxy
**What**: Approximating how long people wait to update their details after initial enrolment.

**Why**: A short latency might indicate poor quality of the initial enrolment (e.g., errors made by operators), forcing citizens to return quickly for corrections.

In [14]:
print("\\n--- Insight 11: Update Latency Proxy ---")
print("Note: This is a high-level proxy. It correlates spikes, but doesn't track individuals.")

# Correlate the time series of enrolments with a lagged time series of updates
latency_df = master_df.groupby('date').agg(
    total_enrolment=('total_enrolment', 'sum'),
    total_updates=('total_updates', 'sum')
).reset_index().sort_values('date')

correlations = {}
for lag in range(1, 31): # Check for lags from 1 to 30 days
    correlations[lag] = latency_df['total_enrolment'].corr(latency_df['total_updates'].shift(-lag))

# Find the lag with the highest correlation
if correlations:
    best_lag = max(correlations, key=correlations.get)
    print(f"Highest correlation found at a lag of {best_lag} days (Correlation: {correlations[best_lag]:.2f}).")
    print("This suggests a potential spike in updates roughly a month after a spike in enrolments.")
else:
    print("Could not compute latency correlation.")

\n--- Insight 11: Update Latency Proxy ---
Note: This is a high-level proxy. It correlates spikes, but doesn't track individuals.
Highest correlation found at a lag of 1 days (Correlation: 0.33).
This suggests a potential spike in updates roughly a month after a spike in enrolments.


### 1️⃣2️⃣ Long-Tail District Burden
**What**: A large number of districts that process a very small volume of transactions.

**Why**: Indicates that fixed infrastructure (like permanent centers) may be inefficient in these areas. Mobile units could be a better solution.

In [15]:
print("\\n--- Insight 12: Long-Tail District Burden ---")
# Define a minimal daily threshold
MINIMAL_DAILY_THRESHOLD = 10
daily_avg = master_df.groupby(['state', 'district'])['total_transactions'].mean().reset_index(name='avg_daily_transactions')

long_tail_districts = daily_avg[daily_avg['avg_daily_transactions'] < MINIMAL_DAILY_THRESHOLD]

print(f"Found {len(long_tail_districts)} 'long-tail' districts with an average of fewer than {MINIMAL_DAILY_THRESHOLD} transactions per day.")
display(long_tail_districts.sort_values('avg_daily_transactions').head(10))

\n--- Insight 12: Long-Tail District Burden ---
Found 15 'long-tail' districts with an average of fewer than 10 transactions per day.


,state,district,avg_daily_transactions
0,Andaman And Nicobar Islands,Nicobars,1.000000
33,Arunachal Pradesh,Leparada,1.555556
659,Uttar Pradesh,Mahrajganj,1.875000
528,Sikkim,Mangan,2.000000
417,Mizoram,Khawzawl,2.266667
416,Mizoram,Hnahthial,2.294118
213,Himachal Pradesh,Lahaul And Spiti,2.636364
397,Manipur,Pherzawl,3.000000
40,Arunachal Pradesh,Pakke Kessang,3.782609
522,Rajasthan,Salumbar,4.181818


### 1️⃣3️⃣ Data Reliability Heatmap
**What**: Where the data cleaning pipeline had to drop the most rows.

**Why**: This is a meta-insight into the trustworthiness of the analytics. Areas with high data loss may have systemic data entry problems, and insights from these areas should be treated with caution.

In [16]:
print("\\n--- Insight 13: Data Reliability Heatmap ---")
def load_log_files(logs_dir: Path) -> pd.DataFrame:
    log_files = list(logs_dir.glob("*_unresolved.csv"))
    if not log_files: return pd.DataFrame()
    log_dfs = [pd.read_csv(file) for file in log_files]
    return pd.concat(log_dfs, ignore_index=True)

dropped_rows_df = load_log_files(LOGS_DIR)

if not dropped_rows_df.empty:
    # Group by the raw state and district names from the logs
    quality_map = dropped_rows_df.groupby(['raw_state', 'raw_district']).size().reset_index(name='dropped_rows_count')
    
    print(f"Total rows dropped by LGD validation: {len(dropped_rows_df):,}")
    print("Top 15 locations with the most dropped rows (potential data quality issues):")
    display(quality_map.sort_values('dropped_rows_count', ascending=False).head(15))
else:
    print("No dropped row logs were found. Data quality appears to be high or logs are missing.")

\n--- Insight 13: Data Reliability Heatmap ---
Total rows dropped by LGD validation: 701,669
Top 15 locations with the most dropped rows (potential data quality issues):


,raw_state,raw_district,dropped_rows_count
228,West Bengal,Barddhaman,28256
95,Karnataka,Bengaluru,23752
92,Karnataka,Bangalore,19764
94,Karnataka,Belgaum,18905
126,Maharashtra,Ahmadnagar,18831
3,Andhra Pradesh,Anantapur,16821
188,Tamil Nadu,Tiruvallur,16061
5,Andhra Pradesh,Cuddapah,15751
9,Andhra Pradesh,Nellore,15520
113,Kerala,Pathanamthitta,15051


### 1️⃣4️⃣ Administrative Boundary Drift
**What**: Old district names that are no longer in the official Local Government Directory (LGD) but still appear frequently in historical data.

**Why**: Signals a lag in administrative systems updating to reflect new district boundaries or name changes.

In [17]:
print("\\n--- Insight 14: Administrative Boundary Drift ---")
# This requires the raw data and a list of current, valid districts
if not raw_df.empty:
    current_valid_districts = set(master_df['district'].unique())
    raw_districts = set(raw_df['district'].str.title().unique())
    
    # Find districts in raw data that are NOT in the cleaned, valid set
    legacy_districts = raw_districts - current_valid_districts
    
    if legacy_districts:
        print(f"Found {len(legacy_districts)} potential legacy district names that are no longer in the LGD.")
        # Count their frequency in the raw data
        legacy_counts = raw_df[raw_df['district'].str.title().isin(legacy_districts)]['district'].value_counts().reset_index()
        legacy_counts.columns = ['legacy_district_name', 'frequency']
        display(legacy_counts.head(10))
    else:
        print("No significant administrative boundary drift detected.")
else:
    print("Raw data not found. Skipping boundary drift analysis.")

\n--- Insight 14: Administrative Boundary Drift ---
Found 285 potential legacy district names that are no longer in the LGD.


,legacy_district_name,frequency
0,Barddhaman,28256
1,Bengaluru,23752
2,Bangalore,19764
3,K.v. Rangareddy,19701
4,Belgaum,18905
5,Ahmadnagar,18831
6,Anantapur,16821
7,Villupuram,16544
8,Tiruvallur,16061
9,Cuddapah,15751


### 1️⃣5️⃣ Cross-State Misattribution Frequency
**What**: Districts that are repeatedly tagged to the wrong state in the raw data.

**Why**: Points to systemic data entry errors or confusing UI design in the enrolment software.

In [18]:
print("\\n--- Insight 15: Cross-State Misattribution Frequency ---")
if not dropped_rows_df.empty:
    # Filter logs for rows dropped due to state mismatch
    mismatch_df = dropped_rows_df[dropped_rows_df['drop_reason'] == 'district_belongs_to_other_state']
    
    if not mismatch_df.empty:
        mismatch_counts = mismatch_df.groupby(['raw_state', 'raw_district', 'normalized_state']).size().reset_index(name='mismatch_count')
        print(f"Found {len(mismatch_counts)} instances of district-state mismatches.")
        print("The table shows the incorrect raw state and the correct normalized state.")
        display(mismatch_counts.sort_values('mismatch_count', ascending=False).head(10))
    else:
        print("No cross-state misattributions found in the logs.")
else:
    print("Log files not available for misattribution analysis.")

\n--- Insight 15: Cross-State Misattribution Frequency ---
No cross-state misattributions found in the logs.


### 1️⃣6️⃣ Temporal Adoption Curve
**What**: Whether a district's transaction volume is growing, stagnating, or declining over time.

**Why**: Provides a long-term planning perspective, helping to allocate future resources.

In [19]:
print("\\n--- Insight 16: Temporal Adoption Curve ---")
master_df['month_ordinal'] = master_df['date'].apply(lambda x: x.toordinal())

def get_slope(df):
    # Need at least 3 points to calculate a meaningful slope
    if len(df) < 3:
        return np.nan
    slope, _, _, _, _ = linregress(df['month_ordinal'], df['total_transactions'])
    return slope

# Group by district and apply the regression
adoption_curves = master_df.groupby(['state', 'district']).apply(get_slope).reset_index(name='slope')

# Categorize based on slope
growing_districts = adoption_curves[adoption_curves['slope'] > adoption_curves['slope'].quantile(0.75)]
declining_districts = adoption_curves[adoption_curves['slope'] < adoption_curves['slope'].quantile(0.25)]

print(f"Found {len(growing_districts)} districts with a strong growth trend.")
display(growing_districts.sort_values('slope', ascending=False).head(5))

print(f"\\nFound {len(declining_districts)} districts with a strong declining trend.")
display(declining_districts.sort_values('slope').head(5))

\n--- Insight 16: Temporal Adoption Curve ---
Found 179 districts with a strong growth trend.


,state,district,slope
428,Nagaland,Meluri,0.082860
263,Karnataka,Bengaluru South,0.080812
493,Rajasthan,Balotra,0.042583
497,Rajasthan,Beawar,0.021352
366,Maharashtra,Gondia,0.019346


\nFound 179 districts with a strong declining trend.


,state,district,slope
386,Maharashtra,Thane,-128.608135
379,Maharashtra,Pune,-125.648796
154,Gujarat,Ahmedabad,-112.962331
181,Gujarat,Surat,-111.554301
510,Rajasthan,Jaipur,-94.739530


### 1️⃣7️⃣ Processing Consistency Index
**What**: Compares the variance in transaction volume on the same day of the week (e.g., all Mondays vs. all Tuesdays).

**Why**: Can act as a proxy for operator or center efficiency. High variance on a specific weekday might indicate inconsistent staffing or recurring technical issues.

In [20]:
print("\\n--- Insight 17: Processing Consistency Index ---")
master_df['weekday'] = master_df['date'].dt.day_name()
# Variance of transactions for each day of the week, per district
consistency_df = master_df.groupby(['state', 'district', 'weekday'])['total_transactions'].var().unstack().reset_index().fillna(0)

# Calculate a simple consistency score (lower is better)
weekday_cols = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
consistency_df['consistency_score'] = consistency_df[weekday_cols].mean(axis=1)

print("Top 10 districts with the most inconsistent processing (highest variance):")
display(consistency_df.sort_values('consistency_score', ascending=False).head(10))

\n--- Insight 17: Processing Consistency Index ---
Top 10 districts with the most inconsistent processing (highest variance):


weekday,state,district,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday,consistency_score
386,Maharashtra,Thane,2.372189e+09,2.460858e+09,6.948642e+08,9.938945e+08,9860346.7,1.484365e+09,1.327859e+07,1.147044e+09
379,Maharashtra,Pune,3.206044e+09,1.602560e+09,8.104708e+08,5.957486e+08,17381721.3,1.661266e+09,1.555556e+07,1.129861e+09
154,Gujarat,Ahmedabad,1.986409e+09,1.025934e+09,6.831037e+08,9.806629e+08,2004499.7,8.077855e+08,4.085181e+06,7.842835e+08
181,Gujarat,Surat,1.474723e+09,7.055303e+08,6.853152e+08,1.443740e+09,809969.3,9.025191e+08,2.778176e+06,7.450594e+08
510,Rajasthan,Jaipur,1.538510e+09,5.605619e+08,7.821461e+08,4.402846e+08,1467413.5,7.810525e+08,2.290616e+06,5.866161e+08
376,Maharashtra,Nashik,1.337275e+09,1.017965e+09,2.483544e+08,3.261744e+08,19001812.3,1.148212e+09,9.198976e+06,5.865974e+08
13,Andhra Pradesh,Kurnool,8.407998e+08,8.882210e+08,9.933333e+08,6.540133e+08,598334.8,2.507449e+08,1.855217e+06,5.185094e+08
571,Telangana,Hyderabad,8.499553e+08,7.620850e+08,3.049596e+08,2.717898e+08,1070660.5,8.919235e+08,2.959167e+06,4.406776e+08
362,Maharashtra,Chhatrapati Sambhajinagar,1.406162e+09,5.395030e+08,2.551232e+08,2.200935e+08,2704779.0,5.135435e+08,9.445687e+06,4.209393e+08
22,Andhra Pradesh,Visakhapatnam,9.238439e+08,7.405172e+08,7.014324e+08,3.897802e+08,457108.7,1.026868e+08,2.029035e+06,4.086781e+08


### 1️⃣8️⃣ Holiday Sensitivity Score
**What**: The drop in transaction volume during weekends/holidays compared to weekdays.

**Why**: Highlights dependency on standard working hours and identifies areas where 24/7 or extended-hour services might be beneficial.

In [21]:
print("\\n--- Insight 18: Holiday Sensitivity Score ---")
master_df['is_weekend'] = master_df['date'].dt.dayofweek >= 5 # Saturday=5, Sunday=6
weekend_vs_weekday = master_df.groupby(['state', 'district', 'is_weekend'])['total_transactions'].mean().unstack().reset_index().fillna(0)
weekend_vs_weekday.columns = ['state', 'district', 'weekday_avg', 'weekend_avg']

weekend_vs_weekday['holiday_sensitivity_score'] = (weekend_vs_weekday['weekday_avg'] - weekend_vs_weekday['weekend_avg']) / (weekend_vs_weekday['weekday_avg'] + 1)

# High score means a big drop on weekends
highly_sensitive = weekend_vs_weekday[weekend_vs_weekday['holiday_sensitivity_score'] > 0.9]

print(f"Found {len(highly_sensitive)} districts that are highly sensitive to holidays/weekends.")
display(highly_sensitive.sort_values('holiday_sensitivity_score', ascending=False).head(10))

\n--- Insight 18: Holiday Sensitivity Score ---
Found 11 districts that are highly sensitive to holidays/weekends.


,state,district,weekday_avg,weekend_avg,holiday_sensitivity_score
264,Karnataka,Bengaluru Urban,7530.000000,0.000000,0.999867
108,Bihar,Purbi Champaran,4908.333333,0.000000,0.999796
199,Haryana,Nuh,1506.000000,0.000000,0.999336
305,Madhya Pradesh,Ashoknagar,1505.500000,0.000000,0.999336
242,Jharkhand,East Singhbum,1447.000000,0.000000,0.999309
191,Haryana,Gurugram,878.666667,0.000000,0.998863
461,Odisha,Nabarangpur,442.000000,0.000000,0.997743
234,Jammu And Kashmir,Shopian,412.000000,0.000000,0.997579
77,Assam,Sivasagar,211.000000,0.000000,0.995283
262,Karnataka,Bengaluru Rural,217.846154,1.666667,0.987815


### 1️⃣9️⃣ Over-Centralization Risk
**What**: The degree to which a state's capital or primary metro district dominates its total Aadhaar transaction volume.

**Why**: High centralization can indicate neglect of rural or remote areas, forcing citizens to travel long distances for services.

In [22]:
print("\\n--- Insight 19: Over-Centralization Risk ---")
# This is the same as State Dependency Risk, but we can try to identify capital districts
# This requires a list of capitals, which we don't have. We'll use the top district as a proxy.
top_district_per_state = district_dependency.loc[district_dependency.groupby('state')['pct_of_state_total'].idxmax()]

# Flag states where the top district has a very high share (e.g., > 30%)
over_centralized_states = top_district_per_state[top_district_per_state['pct_of_state_total'] > 30]

print(f"Found {len(over_centralized_states)} states where a single district dominates transaction volume (>30%).")
display(over_centralized_states[['state', 'district', 'pct_of_state_total']].sort_values('pct_of_state_total', ascending=False))

\n--- Insight 19: Over-Centralization Risk ---
Found 10 states where a single district dominates transaction volume (>30%).


,state,district,pct_of_state_total
120,Chandigarh,Chandigarh,100.000000
302,Ladakh,Kargil,100.000000
1,Andaman And Nicobar Islands,North And Middle Andaman,99.924562
529,Sikkim,Namchi,83.177570
598,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,74.589813
469,Puducherry,Puducherry,68.039793
152,Goa,North Goa,54.164467
148,Delhi,North East,44.296256
608,Tripura,West Tripura,31.736731
414,Mizoram,Aizawl,31.406899


### 2️⃣0️⃣ Infrastructure Saturation Warning
**What**: Districts that appear to be hitting a hard ceiling on their daily processing capacity.

**Why**: Indicates that the existing infrastructure is maxed out. Even with higher demand, the centers cannot process more applications, leading to long wait times.

In [23]:
print("\\n--- Insight 20: Infrastructure Saturation Warning ---")
# Look for districts where the daily max transaction count is flat across multiple months
master_df['year_month'] = master_df['date'].dt.to_period('M')
monthly_maxes = master_df.groupby(['state', 'district', 'year_month'])['total_transactions'].max().reset_index()

# Calculate the variance of these monthly maximums for each district
saturation_warning = monthly_maxes.groupby(['state', 'district'])['total_transactions'].var().reset_index(name='variance_of_monthly_max')

# A very low variance suggests the max is always the same (a flat ceiling)
# We also need to ensure the max itself is a reasonably high number
avg_max = monthly_maxes.groupby(['state', 'district'])['total_transactions'].mean().reset_index(name='avg_monthly_max')
saturation_warning = pd.merge(saturation_warning, avg_max, on=['state', 'district'])

# Flag districts with low variance and a high average max
saturated_infra = saturation_warning[
    (saturation_warning['variance_of_monthly_max'] < saturation_warning['variance_of_monthly_max'].quantile(0.10)) &
    (saturation_warning['avg_monthly_max'] > saturation_warning['avg_monthly_max'].quantile(0.75))
]

print(f"Found {len(saturated_infra)} districts showing signs of infrastructure saturation.")
display(saturated_infra.sort_values('variance_of_monthly_max').head(10))

\n--- Insight 20: Infrastructure Saturation Warning ---
Found 0 districts showing signs of infrastructure saturation.


,state,district,variance_of_monthly_max,avg_monthly_max
